<a href="https://colab.research.google.com/github/tqd3pz/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [33]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [34]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print('Total revenue:', total_revenue)
print('Total units:', total_units)
print(f'The 400 orders generated ${total_revenue:.2f} in revenue from {total_units} total units sold.')

Total revenue: 8520.0
Total units: 783
The 400 orders generated $8520.00 in revenue from 783 total units sold.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [35]:
by_category = (df.groupby('category')
                    .agg(revenue=('revenue', 'sum'))
                    .sort_values('revenue', ascending=False))

by_category['share_pct'] = (
    100 * by_category['revenue'] / total_revenue
).round(1)

print(by_category)
print('Food generated the most revenue, accounting for 50.4% of total revenue.')

by_category

          revenue  share_pct
category                    
Food       4293.0       50.4
Merch      1771.5       20.8
Drink      1554.0       18.2
RainGear    901.5       10.6
Food generated the most revenue, accounting for 50.4% of total revenue.


,revenue,share_pct
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [36]:
by_vendor = (df.groupby('vendor_id')
                  .agg(avg_order_revenue=('revenue', 'mean'),
                       order_count=('vendor_id', 'count'))
                  .round(2)
                  .sort_values('avg_order_revenue', ascending=False))

print(by_vendor)
print('V-01 has the highest average order revenue at $22.60 across 94 orders.')

by_vendor

           avg_order_revenue  order_count
vendor_id                                
V-01                   22.60           94
V-18                   21.75          108
V-05                   20.58           93
V-10                   20.31          105
V-01 has the highest average order revenue at $22.60 across 94 orders.


,avg_order_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [37]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = 100 * merch_revenue / df['revenue'].sum()

print(f'Merch accounts for {merch_share:.1f}% of total revenue.')

Merch accounts for 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [38]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

before_rows = len(df)
before_revenue = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one',
    indicator=True
)

print('Rows before:', before_rows)
print('Rows after:', len(joined))
print('Revenue before:', before_revenue)
print('Revenue after:', joined['revenue'].sum())

unmatched = joined[joined['_merge'] == 'left_only']

print('Unmatched vendor:', unmatched['vendor_id'].unique())
print('Unmatched revenue:', unmatched['revenue'].sum())

assert len(joined) == before_rows
assert joined['revenue'].sum() == before_revenue

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined = joined.drop(columns='_merge')

Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0
Unmatched vendor: ['V-18']
Unmatched revenue: 2349.0


**The unmatched vendor, and what I did about it: v-18 wasn't in the vendor lookup. i kept its orders and labeled it 'unknown vendor' so its $2,349 in revenue wouldn't be lost.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [39]:
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(pivot)
print('The pivot table shows revenue by vendor and category, with row and column totals.')

pivot

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown vendor    582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0
The pivot table shows revenue by vendor and category, with row and column totals.


category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [40]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. Next game, I would recommend the vendors plan for more demand in the food category, since food generated 4,293 dollars in revenue, or about 50.4 percent of total revenue. In comparison, RainGear only generated 901.50 dollars, or about 10.6 percent of total revenue, so vendors probably wouldn't need to stock as much rain gear unless the weather forecast suggests it will be needed. overall, i would prioritize food inventory because it made up the largest share of the $8,520 total venue.

b. I think q6 is the least trustworthy because one vendor ID, v-18, was missing from the vendor-name lookup.  I know that v-18 accounted for $2,349 in revenue, but I do not know the actual vendor name associated with that ID. this makes the vendor-name comparison less complete, even though the revenue numbers themselves are still correct.